## Environment Initialization and Dependency Conflict Resolution
As default behaviour, Colab pre-loads the latest version of `numpy` into memory (within `sys.modules`) as soon as the runtime starts. However, the `larq` module (and its associated *Keras2/TensorFlow* stack) strictly requires a `numpy` version < 2.0** (v1.26.x).

Executing a standard `%pip install larq` triggers a `numpy` downgrade that crashes the Colab dependency resolver. Since the binaries of the previous version are already allocated in the interpreter's RAM, the system hangs and forces the user to manually restart the runtime.

By introducing the following cell at the beginning of the notebook, we avoid the manual restart of the runtime: it automates the conflict resolution without requiring any manual restart.
1. **Background Installation:** It uses low-level `subprocess` to force the `numpy` downgrade and `larq` installation.
2. **Python Cache Cleanup (`sys.modules`):** It removes references to the old `numpy` modules already loaded in memory. This forces Python to import the new compatible binaries (v1.26) from scratch on the next call, eliminating the need to restart the session.

In [ ]:
import os
import sys
import time
import subprocess

# executing background pip installation for required packages
print("Updating packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "larq", "numpy<2.0", "matplotlib<3.9", "terminaltables", "tqdm", "tensorflow", "-q"])
print("Libraries updated successfully.")

# forcing runtime re-boot if numpy or matplotlib still have the wrong version
try:
    import numpy
    import matplotlib
    if numpy.__version__.startswith('2') or matplotlib.__version__.startswith('3.9'):
        print("Libraries still have the wrong version in memory: rebooting runtime...")
        print(" --> Please, execute again this cell after an error message pops-up in the browser window!")
        print(" --> You can simply close the error message.")

        time.sleep(1)
        os.kill(os.getpid(), 9)
except ImportError:
    pass

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import gc
import random
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ============================== #
# KERAS 2 COMPATIBILITY FOR LARQ #
# ============================== #
USE_KERAS_VERSION = 2
if USE_KERAS_VERSION == 2:
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
else:
    os.environ.pop("TF_USE_LEGACY_KERAS", None)

# ============================ #
# TENSORFLOW/KERAS/LARQ IMPORT #
# ============================ #
import tensorflow as tf
from tensorflow.keras.optimizers import SGD                                                 # type: ignore
from tensorflow.keras.regularizers import l2                                                # type: ignore
from tensorflow.keras.models import Sequential                                              # type: ignore
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense    # type: ignore
from tensorflow.keras.losses import SparseCategoricalCrossentropy                           # type: ignore
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping    # type: ignore

from tqdm.keras import TqdmCallback

from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# ===================== #
# REPRODUCIBILITY SEEDS #
# ===================== #
SEEDS = [1, 42, 1234, 1809, 2602]

def initialize_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

tf.config.experimental.enable_op_determinism()
os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# ==================== #
# DEVICE CONFIGURATION #
# ==================== #
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  print(f"--> Using GPU: {gpus[0].name}")
else:
  print(f"--> Using CPU. No GPU detected.")

# =============== #
# DIRECTORY SETUP #
# =============== #
platforms = ['/content/drive/MyDrive/SEAI_Project', './']    # [colab, windows]
schedulers = ['adaptive_lr', 'lr_1e-1', 'lr_1e-2', 'lr_1e-3', 'lr_1e-4', 'lr_1e-5', 'lr_1e-6']
model_levels = ['baseline', 'constrained', 'quantized_16', 'quantized_8', 'quantized_4', 'quantized_2']
neg_samples_mode = ['clean_neg_samples', 'NOT_clean_neg_samples']

current_platform = ""
current_schedulers = ""
current_model_level = ""
current_neg_samples_mode = ""

input_dir = ""

accuracy_dir = ""
accuracy_graphs_dir = ""
loss_dir = ""
loss_graphs_dir = ""
weights_dir = ""
cm_dir = ""
cm_graphs_dir = ""
roc_dir = ""
roc_graphs_dir = ""

comparison_dir = ""

def create_paths(seed, early_stopping=False, regularization=False):
    global input_dir, accuracy_dir, accuracy_graphs_dir, comparison_dir, loss_dir, loss_graphs_dir, weights_dir, cm_dir, cm_graphs_dir, roc_dir, roc_graphs_dir

    input_dir = f'{current_platform}/input_cnn/scenarios17_21_{current_neg_samples_mode}_60_20_20'

    if not early_stopping:
        accuracy_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/accuracy'
        accuracy_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/accuracy/graphs'
        loss_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/loss"
        loss_graphs_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/loss/graphs"
        weights_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/weights'
        cm_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/cm'
        cm_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/cm/graphs'
        roc_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/roc'
        roc_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/basic/seed_{seed}/roc/graphs'
    else:
        if not regularization:
            accuracy_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/accuracy'
            accuracy_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/accuracy/graphs'
            loss_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/loss"
            loss_graphs_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/loss/graphs"
            weights_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/weights'
            cm_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/cm'
            cm_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/cm/graphs'
            roc_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/roc'
            roc_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/early_stopping/seed_{seed}/roc/graphs'
        else:
            accuracy_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/accuracy'
            accuracy_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/accuracy/graphs'
            loss_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/loss"
            loss_graphs_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/loss/graphs"
            weights_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/weights'
            cm_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/cm'
            cm_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/cm/graphs'
            roc_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/roc'
            roc_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/regularization/seed_{seed}/roc/graphs'

    comparison_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_schedulers}/{current_model_level}/comparison'

    directories = [
        accuracy_dir,
        accuracy_graphs_dir,
        loss_dir,
        loss_graphs_dir,
        weights_dir,
        cm_dir,
        cm_graphs_dir,
        roc_dir,
        roc_graphs_dir,
        comparison_dir
    ]

    for directory in directories:
        os.makedirs(directory, exist_ok=True)

## Dataset and Dataloaders definition

In [ ]:
print("Dataloaders function definition...")
def prepare_datasets(train_path, val_path, test_path, pct=1.0):
    # --------------- #
    # DATASET LOADING #
    # --------------- #
    train_data, val_data, test_data = np.load(train_path), np.load(val_path), np.load(test_path)
    train_x, train_y = train_data['features'], train_data['labels']
    val_x, val_y = val_data['features'], val_data['labels']
    test_x, test_y = test_data['features'], test_data['labels']

    total_train_samples = len(train_x)
    total_val_samples = len(val_x)
    total_test_samples = len(test_x)

    # ----------------------- #
    # PERCENTILE SUB-SAMPLING #
    # ----------------------- #
    pct_train_end_idx = int(len(train_x) * pct)
    pct_val_end_idx = int(len(val_x) * pct)
    pct_test_end_idx = int(len(test_x) * pct)
    train_x, train_y = train_x[:pct_train_end_idx], train_y[:pct_train_end_idx]
    val_x, val_y = val_x[:pct_val_end_idx], val_y[:pct_val_end_idx]
    test_x, test_y = test_x[:pct_test_end_idx], test_y[:pct_test_end_idx]

    pct_train_samples = len(train_x)
    pct_val_samples = len(val_x)
    pct_test_samples = len(test_x)

    # --------------------- #
    # Z-SCORE NORMALIZATION #
    # --------------------- #
    mean_train = np.mean(train_x)
    std_train = np.std(train_x)

    if std_train > 0:
        train_x = (train_x - mean_train) / std_train
        val_x = (val_x - mean_train) / std_train
        test_x = (test_x - mean_train) / std_train

    # ---------- #
    # RE-SHAPING #
    # ---------- #
    if len(train_x.shape) == 3:
        train_x = np.expand_dims(train_x, axis=-1)
        val_x = np.expand_dims(val_x, axis=-1)
        test_x = np.expand_dims(test_x, axis=-1)

    return (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples)


def get_dataset(train_path, val_path, test_path, seed, pct=1.0, batch_size=64):
    (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples) = prepare_datasets(
        train_path, val_path, test_path, pct
    )

    train_set = tf.data.Dataset.from_tensor_slices((train_x, train_y))
    train_set_batched = train_set.shuffle(buffer_size=pct_train_samples, seed=seed).batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    val_set = tf.data.Dataset.from_tensor_slices((val_x, val_y))
    val_set_batched = val_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    test_set = tf.data.Dataset.from_tensor_slices((test_x, test_y))
    test_set_batched = test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    return (train_set_batched, total_train_samples, pct_train_samples), (val_set_batched, total_val_samples, pct_val_samples), (test_set_batched, total_test_samples, pct_test_samples)

print("Successfully defined.")

## CNN Model definition

In [ ]:
print("Baseline model function definition...")
def build_baseline_model():
    model = Sequential(name=f"CNN_Baseline")

    model.add(Input(shape=(16, 54, 1)))                                                                 # input.shape = 16 x 54 x 1 = 864

    # ------- #
    # STACK 1 #
    # ------- #
    model.add(Conv2D(filters=4, kernel_size=(3,3), padding='same', activation='relu', name='conv1'))    # output.shape = 16 x 54 x 4 = 3456
    model.add(Conv2D(filters=4, kernel_size=(3,3), padding='same', activation='relu', name='conv2'))    # output.shape = 16 x 54 x 4 = 3456
    model.add(MaxPooling2D(pool_size=(2,3), name='maxpool1'))                                           # output.shape = 8 x 18 x 4 = 576

    # ------- #
    # STACK 2 #
    # ------- #
    model.add(Conv2D(filters=8, kernel_size=(3,3), padding='same', activation='relu', name='conv3'))    # output.shape = 8 x 18 x 8 = 1152
    model.add(Conv2D(filters=16, kernel_size=(3,3), padding='same', activation='relu', name='conv4'))   # output.shape = 8 x 18 x 16 = 2304
    model.add(MaxPooling2D(pool_size=(2,3), name='maxpool2'))                                           # output.shape = 4 x 6 x 16 = 384

    # -------------------- #
    # PREDICTION COMPONENT #
    # -------------------- #
    model.add(Flatten(name='flatten'))                                                                  # output.shape = 384
    model.add(Dropout(rate=0.2, name='dropout'))                                                        # output.shape = 384

    model.add(Dense(units=2, activation='softmax', name='fc_output'))

    return model

print("Successfully defined.")

## Training and Testing cycles definition

In [ ]:
print("Training and testing cycles definition...")
def train_and_evaluate(weights_dir, scheduler_index, tp, seed, pct=1.0, early_stopping=False, regularization=False):
    global current_schedulers

    pct_formatted = f"{pct * 100:.1f}"
    lr = 1e-2 if scheduler_index == 0 else 1e-1 if scheduler_index == 1 else 1e-2 if scheduler_index == 2 else 1e-3 if scheduler_index == 3 else 1e-4 if scheduler_index == 4 else 1e-5 if scheduler_index == 5 else 1e-6
    current_schedulers = schedulers[scheduler_index]

    train_path = os.path.join(input_dir, 'train', f'train_Tp{tp}.npz')
    val_path = os.path.join(input_dir, 'val', f'val_Tp{tp}.npz')
    test_path = os.path.join(input_dir, 'test', f'test_Tp{tp}.npz')
    (train_set_batched, total_train_samples, pct_train_samples), (val_set_batched, total_val_samples, pct_val_samples), (test_set_batched, total_test_samples, pct_test_samples) = get_dataset(
        train_path, val_path, test_path, seed, pct=pct
    )

    # ------------- #
    # LOADING MODEL #
    # ------------- #
    print("------------------------ NEW MODEL ------------------------")
    print(f"Tp={tp}, Percentage={pct_formatted}%, Lr={lr}, Seed={seed}")

    model = None
    if regularization:
        model = build_baseline_regularized_model()
    else:
        model = build_baseline_model()

    for layer in model.layers:
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            new_weights = [tf.keras.initializers.HeNormal(seed=seed)(layer.weights[0].shape)]
            if len(layer.weights) > 1:
                new_weights.append(tf.keras.initializers.Zeros()(layer.weights[1].shape))
            layer.set_weights(new_weights)

    tp_dir = f"Tp_{tp}"
    weights_file = os.path.join(weights_dir, tp_dir, f'weights_Tp{tp}_pct{pct_formatted}.weights.h5')
    if os.path.exists(weights_file):
        print(f"Model already exists!")
        print(f" --> Loading model from: {weights_file}")
        model.load_weights(weights_file)

    model.compile(
        optimizer=SGD(learning_rate=lr, momentum=0.9, nesterov=True),
        loss=SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    # -------------- #
    # TRAINING MODEL #
    # -------------- #
    if not os.path.exists(weights_file):
        print(f"Training model...")
        print(f" --> Dataset statistics | Total samples: {total_train_samples + total_val_samples + total_test_samples}, Pct total samples: {pct_train_samples + pct_val_samples + pct_test_samples}, Pct train samples: {pct_train_samples}, Pct val samples: {pct_val_samples}, Pct test samples: {pct_test_samples}")
        os.makedirs(os.path.dirname(weights_file), exist_ok=True)

        # Callbacks #
        tqdm_cb = TqdmCallback(verbose=0)

        early_stopping_cb = EarlyStopping(
            monitor='val_loss',
            patience=20,
            min_delta=0.0,
            verbose=0,
            restore_best_weights=True
        )

        checkpoint_cb = ModelCheckpoint(
            filepath=os.path.join(weights_file),
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=True,
            verbose=0
        )

        lr_scheduler = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=50,
            min_lr=1e-6,
            verbose=0
        )

        callbacks_list = [tqdm_cb]
        if current_schedulers == schedulers[0]:
            callbacks_list.append(lr_scheduler)
        if early_stopping:
            callbacks_list.append(early_stopping_cb)
            callbacks_list.append(checkpoint_cb)

        # Actual training #
        history = None
        history = model.fit(
            train_set_batched,
            epochs=2000,
            validation_data=val_set_batched,
            callbacks=callbacks_list,
            verbose=0
        )

        if not early_stopping:
            model.save_weights(weights_file)

        # Plotting loss and accuracies #
        plot_loss(
            history.history['loss'],
            history.history['val_loss'],
            os.path.join(loss_dir, f"tloss_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(loss_dir, f"vloss_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(loss_graphs_dir, f"loss_Tp{tp}_pct{pct_formatted}.pdf"),
            tp,
            seed,
            early_stopping,
            regularization
        )
        plot_accuracy(
            history.history['accuracy'],
            history.history['val_accuracy'],
            os.path.join(accuracy_dir, f"taccuracy_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(accuracy_dir, f"vaccuracy_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(accuracy_graphs_dir, f"accuracy_Tp{tp}_pct{pct_formatted}.pdf"),
            tp,
            seed,
            early_stopping,
            regularization
        )

    # ---------------- #
    # EVALUATING MODEL #
    # ---------------- #
    print("\nEvaluating model...")
    eval_results = model.evaluate(test_set_batched, verbose=0)
    accuracy = eval_results[1] * 100
    print(f" --> Test Accuracy: {accuracy:.2f}%")

    y_pred_probs = model.predict(test_set_batched)
    y_true = np.concatenate([y for x, y in test_set_batched], axis=0)

    plot_cm(y_pred_probs, y_true, tp, pct_formatted, seed, early_stopping, regularization)
    plot_ROC_curve(y_pred_probs, y_true, tp, pct_formatted, seed, early_stopping, regularization)
    print(f"-----------------------------------------------------------\n\n")

    tf.keras.backend.clear_session()
    gc.collect()

    return model, accuracy

print("Successfully defined.")

## Plotting Training and Validation Loss

In [ ]:
print("Defining loss plot function...")
def plot_loss(train_loss, val_loss, tloss_save_path, vloss_save_path, graph_save_path, tp, seed, early_stopping=False, regularization=False):
    np.save(tloss_save_path, train_loss)
    np.save(vloss_save_path, val_loss)

    epochs = range(1, len(train_loss) + 1)

    plt.figure(figsize=(5, 3.8))
    plt.plot(epochs, train_loss, label='Training Loss', color='blue', linewidth=0.6)
    plt.plot(epochs, val_loss, label='Validation Loss', color='red', linewidth=0.6)

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.suptitle(f'Training vs Validation Loss', fontsize=11, y=0.96, ha='center')

    if early_stopping and regularization:
        plt.title(f'Baseline + EarlyStop + L2-Reg - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    elif early_stopping:
        plt.title(f'Baseline + EarlyStop - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    else:
        plt.title(f'Baseline - Tp={tp} - Seed={seed}', fontsize=9, pad=6)

    plt.subplots_adjust(top=0.85)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()

    plt.savefig(graph_save_path, format='pdf', bbox_inches='tight')
    plt.show()

print("Successfully defined.")

## Plotting Training and Validation Accuracies

In [ ]:
print("Defining accuracies plot function...")
def plot_accuracy(train_accuracy, val_accuracy, taccuracy_save_path, vaccuracy_save_path, graph_save_path, tp, seed, early_stopping=False, regularization=False):
    np.save(taccuracy_save_path, train_accuracy)
    np.save(vaccuracy_save_path, val_accuracy)

    epochs = range(1, len(train_accuracy) + 1)

    plt.figure(figsize=(5, 3.8))
    plt.plot(epochs, train_accuracy, label='Training Accuracy', color='blue', linewidth=0.6)
    plt.plot(epochs, val_accuracy, label='Validation Accuracy', color='red', linewidth=0.6)

    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.suptitle(f'Training VS. Validation Accuracy', fontsize=11, y=0.96, ha='center')

    if early_stopping and regularization:
        plt.title(f'Baseline + EarlyStop + L2-Reg - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    elif early_stopping:
        plt.title(f'Baseline + EarlyStop - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    else:
        plt.title(f'Baseline - Tp={tp} - Seed={seed}', fontsize=9, pad=6)

    plt.subplots_adjust(top=0.85)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()

    plt.savefig(graph_save_path, format='pdf', bbox_inches='tight')
    plt.show()

print("Successfully defined.")

## Plotting Test Confusion Matrix

In [ ]:
print("Defining confusion matrix plot function...")
def plot_cm(y_pred_probs, y_true, tp, pct_formatted, seed, early_stopping=False, regularization=False):
    y_pred_classes = np.argmax(y_pred_probs, axis=1)

    cm = confusion_matrix(y_true, y_pred_classes)
    np.save(os.path.join(cm_dir, f'cm_array_Tp{tp}_pct{pct_formatted}.npy'), cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.suptitle(f'Confusion Matrix', fontsize=11, y=0.96, ha='center')

    if early_stopping and regularization:
        plt.title(f'Baseline + EarlyStop + L2-Reg - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    elif early_stopping:
        plt.title(f'Baseline + EarlyStop - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    else:
        plt.title(f'Baseline - Tp={tp} - Seed={seed}', fontsize=9, pad=6)

    plt.subplots_adjust(top=0.85)

    plt.savefig(os.path.join(cm_graphs_dir, f'cm_Tp{tp}_pct{pct_formatted}.pdf'), format='pdf', bbox_inches='tight')

    plt.show()
    plt.close()

print("Successfully defined.")

## Plotting Test ROC Curve

In [ ]:
print("Defining ROC Curve plot function...")
def plot_ROC_curve(y_pred_probs, y_true, tp, pct_formatted, seed, early_stopping=False, regularization=False):
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs[:, 1])
    np.savez(os.path.join(roc_dir, f'roc_arrays_Tp{tp}_pct{pct_formatted}.npz'), fpr=fpr, tpr=tpr)

    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5, 3.8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC area = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.suptitle(f'ROC Curve', fontsize=11, y=0.96, ha='center')

    if early_stopping and regularization:
        plt.title(f'Baseline + EarlyStop + L2-Reg - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    elif early_stopping:
        plt.title(f'Baseline + EarlyStop - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    else:
        plt.title(f'Baseline - Tp={tp} - Seed={seed}', fontsize=9, pad=6)

    plt.subplots_adjust(top=0.85)
    plt.legend(loc="lower right")

    plt.savefig(os.path.join(roc_graphs_dir, f'roc_Tp{tp}_pct{pct_formatted}.pdf'), format='pdf', bbox_inches='tight')

    plt.show()
    plt.close()

print("Successfully defined.")

## Plotting Test Accuracy Comparison

In [ ]:
print("Defining accuracy plot function...")
# paper_accuracies_tp2 = np.array([
#         58.0, 63.0, 68.0, 70.5, 74.5, 74.8, 75.0, 77.0, 78.0, 78.2,
#         78.5, 78.8, 79.0, 79.3, 79.1, 79.8, 80.2, 80.3, 80.5, 81.5,
#         81.5, 81.5, 81.5, 82.5, 81.8, 82.0, 82.2, 82.4, 81.8, 83.0,
#         82.9, 83.5, 82.6, 84.0, 83.5, 84.2, 83.6, 83.7, 83.7, 83.8
#     ])
# paper_accuracies_tp2_sampled = np.array([58.0, 70.5, 77.0, 78.8, 79.8, 81.5, 82.5, 82.4, 83.5, 84.2, 83.8])
paper_accuracies_100 = np.array([97.0, 86.6, 74.8, 69.3, 68.5, 65.8, 67.0, 65.7, 63.5, 63.7])

# x_axis_pct = np.array([2.5, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0])
x_axis_tp = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# def plot_accuracy_vs_pct_comparison(accuracies_tp2):
#     diff_accuracies_tp2 = np.abs(np.array(accuracies_tp2) - paper_accuracies_tp2_sampled)

#     plt.figure(figsize=(7, 4))

#     plt.plot(x_axis_pct, accuracies_tp2, marker='o', color='navy', label='Our Baseline CNN')
#     plt.plot(x_axis_pct, paper_accuracies_tp2_sampled, marker='x', color='gray', label='Paper CNN')
#     plt.plot(x_axis_pct, diff_accuracies_tp2, marker='^', linestyle='--', color='darkgreen', label='Difference')

#     plt.xlabel('Dataset size (%)')
#     plt.ylabel('Accuracy (%)')
#     plt.suptitle('Accuracy vs Dataset Size', x=0.5, y=0.95, fontsize=14)
#     plt.title(f'Tp = 2 - Lr = {current_schedulers}', fontsize=9, pad=6)
#     plt.grid(True, linestyle='--', alpha=0.4)
#     plt.legend()

#     plt.tight_layout()

#     plt.savefig(os.path.join(comparison_dir, f'comparison_Tp2_vs_dataset_size.pdf'), format='pdf', bbox_inches='tight')
#     plt.show()

#     np.save(os.path.join(comparison_dir, f'testAccuracies_Tp2.npy'), accuracies_tp2)

def plot_accuracy_vs_tp_comparison(accuracies_basic=[], accuracies_early_stopping=[], plot_early_stopping=False, accuracies_regularization=[], plot_regularization=False):
    plt.figure(figsize=(7, 4))

    plt.plot(x_axis_tp, paper_accuracies_100, marker='x', color='gray', label='Paper CNN')
    plt.plot(x_axis_tp, accuracies_basic, marker='o', color='blue', label='Baseline CNN')
    if plot_early_stopping:
        plt.plot(x_axis_tp, accuracies_early_stopping, marker='o', color='steelblue', label='Baseline + EarlyStop')
    if plot_regularization:
        plt.plot(x_axis_tp, accuracies_regularization, marker='o', color='navy', label='Baseline + EarlyStop + L2-Reg')

    plt.xlabel('Tp')
    plt.ylabel('Accuracy (%)')
    plt.suptitle('Test Accuracy vs Tp', x=0.5, y=0.95, fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.tight_layout()

    if not plot_early_stopping and not plot_regularization:
        plt.savefig(os.path.join(comparison_dir, f'testAccuracy_basic_vs_tp.pdf'), format='pdf', bbox_inches='tight')
    elif plot_early_stopping and not plot_regularization:
        plt.savefig(os.path.join(comparison_dir, f'testAccuracy_basic_earlyStopping_vs_tp.pdf'), format='pdf', bbox_inches='tight')
    else:
        plt.savefig(os.path.join(comparison_dir, f'testAccuracy_basic_earlyStopping_regularization_vs_tp.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

    np.save(os.path.join(comparison_dir, f'testAccuracyBasic.npy'), accuracies_basic)
    if plot_early_stopping:
        np.save(os.path.join(comparison_dir, f'testAccuracyEarlyStopping.npy'), accuracies_early_stopping)
    if plot_regularization:
        np.save(os.path.join(comparison_dir, f'testAccuracyRegularization.npy'), accuracies_regularization)

print("Successfully defined.")

## Plotting Test Accuracy Distribution

In [ ]:
def plot_accuracy_distribution_error(tps, accuracy_distributions, mean_accuracies, model_name, scheduler_index, result_dir, base_color='navy', y_window_size=15.0):
  print(f"Generating mean accuracy line plot with min-max error bars for scheduler {scheduler_index}...")

  rgb_color = mcolors.to_rgb(base_color)
  hsv_color = mcolors.rgb_to_hsv(rgb_color)

  hsv_color[2] = max(0.0, hsv_color[2] * 0.7)
  darker_color = mcolors.hsv_to_rgb(hsv_color)

  min_accuracies = [np.min(dist) for dist in accuracy_distributions]
  max_accuracies = [np.max(dist) for dist in accuracy_distributions]

  lower_errors = np.array(mean_accuracies) - np.array(min_accuracies)
  upper_errors = np.array(max_accuracies) - np.array(mean_accuracies)
  error_bounds = [lower_errors, upper_errors]

  plt.figure(figsize=(7, 4))

  plt.errorbar(tps, mean_accuracies, yerr=error_bounds, fmt='-s',
               color=base_color, ecolor=darker_color, elinewidth=2, capsize=6,
               capthick=2, markersize=8, label='Accuracy Distribution', zorder=3)

  local_min = np.min(min_accuracies)
  local_max = np.max(max_accuracies)
  data_span = local_max - local_min

  mid_point = (local_max + local_min) / 2.0

  if data_span > y_window_size:
    margin = 1.0
    y_limits = (local_min - margin, local_max + margin)
  else:
    half_window = y_window_size / 2.0
    y_limits = (mid_point - half_window, mid_point + half_window)


  plt.title(f"Test Accuracy Distribution {model_name.title()} Model vs TP", fontsize=15)
  plt.xlabel("TP", fontsize=13)
  plt.ylabel("Accuracy (%)", fontsize=13)
  plt.xticks(tps)
  plt.grid(axis='y', linestyle='--', alpha=0.7)
  plt.legend(loc='upper right')

  plt.ylim(y_limits)

  plt.tight_layout()

  plt.savefig(os.path.join(result_dir, f'accuracy_trend_lineplot_{model_name}_scheduler{scheduler_index}.pdf'), format='pdf', bbox_inches='tight')
  plt.show()


In [ ]:
def analyze_and_plot_all_models():
  results_dir = os.path.join(current_platform, "experiment_results/")
  file_pattern = os.path.join(results_dir, "*.json")
  result_files = glob.glob(file_pattern)

  if not result_files:
    print(f"No JSON files found in the directory {results_dir}.")
    return

  all_raw_accuracies = []
  loaded_experiments = []

  for file_path in result_files:
    with open(file_path, "r") as f:
      data = json.load(f)
      loaded_experiments.append(data)

      for dist in data["accuracies_by_tp"]:
        all_raw_accuracies.extend(dist)

  for exp_data in loaded_experiments:
    model_name = exp_data.get("model_name", "unknown_model")
    scheduler_index = exp_data.get("scheduler_index", "N/A")

    if "baseline" in model_name.lower():
      color = "navy"
    elif "constrained" in model_name.lower():
      color = "darkorange"
    elif "quantized16" in model_name.lower():
      color = "darkgreen"
    elif "quantized8" in model_name.lower():
      color = "teal"
    elif "quantized4" in model_name.lower():
      color = "purple"
    elif "quantized2" in model_name.lower():
      color = "crimson"
    else:
      color = "indigo"

    plot_accuracy_distribution_error(
        tps=exp_data["tps"],
        accuracy_distributions=exp_data["accuracies_by_tp"],
        mean_accuracies=exp_data["mean_accuracies_by_tp"],
        model_name=model_name,
        scheduler_index=scheduler_index,
        result_dir=results_dir,
        base_color=color
    )

## Main execution

In [ ]:
if __name__ == "__main__":
    current_platform = platforms[0]                     # colab
    current_model_level = model_levels[0]               # baseline
    current_neg_samples_mode = neg_samples_mode[0]      # clean_neg_samples

    scheduler_indeces = [3]

    tps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    # base_pcts = np.r_[0.025, np.arange(0.1, 1.1, 0.1)]

    print("Executing experiment...")
    accuracies = []
    for scheduler_index in scheduler_indeces:
        current_schedulers = schedulers[scheduler_index]

        accuracies_by_tp = []
        mean_accuracies_by_tp = []

        accuracies_by_scheduler = []
        for tp in tps:
            accuracies_by_seed = []
            for seed in SEEDS:
                create_paths(seed)
                initialize_seed(seed)

                _, accuracy = train_and_evaluate(
                    weights_dir=weights_dir,
                    scheduler_index=scheduler_index,
                    tp=tp,
                    seed=seed,
                    pct=1.0
                )

                accuracies_by_seed.append(accuracy)

            accuracies_by_tp.append(accuracies_by_seed)
            mean_accuracy = np.mean(accuracies_by_seed)
            mean_accuracies_by_tp.append(mean_accuracy)

            mean_accuracy_by_tp = np.mean(accuracies_by_seed)
            accuracies_by_scheduler.append(mean_accuracy_by_tp)
        accuracies.append(accuracies_by_scheduler)

## Plot comparison graphs

In [ ]:
# plot_accuracy_vs_pct_comparison(accuracies[0][1])

In [ ]:
plot_accuracy_vs_tp_comparison(np.array(accuracies[0]))

## Training and Testing again with Early Stopping

In [ ]:
if __name__ == "__main__":
    current_platform = platforms[0]                     # colab
    current_model_level = model_levels[0]               # baseline
    current_neg_samples_mode = neg_samples_mode[0]      # clean_neg_samples

    scheduler_indeces = [3]

    tps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    print("Executing experiment with early stopping...")
    accuracies_early_stopping = []
    for scheduler_index in scheduler_indeces:
        current_schedulers = schedulers[scheduler_index]

        accuracies_by_scheduler = []
        for tp in tps:
            accuracies_by_seed = []
            for seed in SEEDS:
                create_paths(seed, early_stopping=True)
                initialize_seed(seed)

                _, accuracy = train_and_evaluate(
                    weights_dir=weights_dir,
                    scheduler_index=scheduler_index,
                    tp=tp,
                    seed=seed,
                    pct=1.0,
                    early_stopping=True
                )

                accuracies_by_seed.append(accuracy)
            mean_accuracy_by_tp = np.mean(accuracies_by_seed)
            accuracies_by_scheduler.append(mean_accuracy_by_tp)
        accuracies_early_stopping.append(accuracies_by_scheduler)

## Plot comparison graphs

In [ ]:
plot_accuracy_vs_tp_comparison(np.array(accuracies[0]), np.array(accuracies_early_stopping[0]), True)

## Introducing L2 Regularization in the CNN Model

In [ ]:
print("Baseline model function definition with regularization...")
def build_baseline_regularized_model():
    model = Sequential(name=f"CNN_Baseline_Regularized")
    reg_lambda = 1e-3

    model.add(Input(shape=(16, 54, 1)))                                                                                                     # input.shape = 16 x 54 x 1 = 864

    # ------- #
    # STACK 1 #
    # ------- #
    model.add(Conv2D(filters=4, kernel_size=(3,3), padding='same', activation='relu', kernel_regularizer=l2(reg_lambda), name='conv1'))     # output.shape = 16 x 54 x 4 = 3456
    model.add(Conv2D(filters=4, kernel_size=(3,3), padding='same', activation='relu', kernel_regularizer=l2(reg_lambda), name='conv2'))     # output.shape = 16 x 54 x 4 = 3456
    model.add(MaxPooling2D(pool_size=(2,3), name='maxpool1'))                                                                               # output.shape = 8 x 18 x 4 = 576

    # ------- #
    # STACK 2 #
    # ------- #
    model.add(Conv2D(filters=8, kernel_size=(3,3), padding='same', activation='relu', kernel_regularizer=l2(reg_lambda), name='conv3'))     # output.shape = 8 x 18 x 8 = 1152
    model.add(Conv2D(filters=16, kernel_size=(3,3), padding='same', activation='relu', kernel_regularizer=l2(reg_lambda), name='conv4'))    # output.shape = 8 x 18 x 16 = 2304
    model.add(MaxPooling2D(pool_size=(2,3), name='maxpool2'))                                                                               # output.shape = 4 x 6 x 16 = 384

    # -------------------- #
    # PREDICTION COMPONENT #
    # -------------------- #
    model.add(Flatten(name='flatten'))                                                                                                      # output.shape = 384
    model.add(Dropout(rate=0.2, name='dropout'))                                                                                            # output.shape = 384

    model.add(Dense(units=2, activation='softmax', kernel_regularizer=l2(reg_lambda), name='fc_output'))

    return model

print("Successfully defined.")

## Training and Testing with L2 Regularization

In [ ]:
if __name__ == "__main__":
    current_platform = platforms[0]                     # colab
    current_model_level = model_levels[0]               # baseline
    current_neg_samples_mode = neg_samples_mode[0]      # clean_neg_samples

    scheduler_indeces = [3]

    tps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    print("Executing experiment with early stopping and L2-regularization...")
    accuracies_regularization = []
    for scheduler_index in scheduler_indeces:
        current_schedulers = schedulers[scheduler_index]

        accuracies_by_tp = []
        mean_accuracies_by_tp = []

        accuracies_by_scheduler = []
        for tp in tps:
            accuracies_by_seed = []
            for seed in SEEDS:
                create_paths(seed)
                initialize_seed(seed)

                _, accuracy = train_and_evaluate(
                    weights_dir=weights_dir,
                    scheduler_index=scheduler_index,
                    tp=tp,
                    seed=seed,
                    pct=1.0
                )

                accuracies_by_seed.append(accuracy)

            accuracies_by_tp.append(accuracies_by_seed)
            mean_accuracy = np.mean(accuracies_by_seed)
            mean_accuracies_by_tp.append(mean_accuracy)

            mean_accuracy_by_tp = np.mean(accuracies_by_seed)
            accuracies_by_scheduler.append(mean_accuracy_by_tp)
        accuracies_regularization.append(accuracies_by_scheduler)

In [ ]:
def save_experiment_results(model_name, scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp):
  output_dir = os.path.join(current_platform, "experiment_results/")
  os.makedirs(output_dir, exist_ok=True)

  experiment_data = {
      "model_name": model_name,
      "scheduler_index": scheduler_index,
      "tps": tps,
      "accuracies_by_tp" : accuracies_by_tp,
      "mean_accuracies_by_tp": mean_accuracies_by_tp
  }

  output_file = os.path.join(output_dir, f"results_{model_name}_scheduler{scheduler_index}.json")

  with open(output_file, "w") as f:
    json.dump(experiment_data, f, indent=4)
  print(f"Data saved to  {output_file}")

In [ ]:
save_experiment_results("baseline", scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp)

## Plot comparison graphs

In [ ]:
plot_accuracy_vs_tp_comparison(np.array(accuracies[0]), np.array(accuracies_early_stopping[0]), True, np.array(accuracies_regularization[0]), True)

In [ ]:
plot_accuracy_distribution_error(tps, accuracies_by_tp, mean_accuracies_by_tp, "baseline", scheduler_index, os.path.join(current_platform, "experiment_results/"))

In [ ]:
analyze_and_plot_all_models()